In [1]:
import numpy as np
import keras as kp
import pandas as pd

In this notebook I plan to use Keras to read data from CDC's nutrition survey, to find correlations between dietary habits and high blood pressure.
The data can be found here: https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2017
From this data, I am pulling the dietary data: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Dietary&CycleBeginYear=2017,
Specifically choosing the dietary and nutritional data for the first day of the survey, labeled:
Dietary Interview - Individual Foods, First Day
Dietary Interview - Total Nutrient Intakes, First Day

As well as the blood pressure info from the examination data: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Examination&CycleBeginYear=2017
The demographics data: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Demographics&CycleBeginYear=2017
And physical activity info from the general survey questions: data: https://wwwn.cdc.gov/nchs/nhanes/search/datapage.aspx?Component=Questionnaire&CycleBeginYear=2017

My goal is to use Keras to find possible associations between the dietary, nutritional, and aerobic exercise habits of the test takers with the results of their blood pressure panels. 

Limits:
What I am trying to do is find a real assocation between people's daily habits and their health, in the real world. This survey, and the specific data I am pulling from it, constitutes a single day of data-taking, and does not accurately reflect the real world to an extent. I have to assume that it is possible that someone's typical daily habits have effected their blood pressure in a way that is not accurately reflected in that single day of data collection. I am also relying on test-takers to reliably and accurately record their own habits. Despite this, considering the large number of survey takers, and the resistance average blood pressure has against immediate (non cumulative) changes, I expect to find some positive correlations between daily habits and blood pressure. I can further test this data against real-world imperical research to test the accuracy of the model I develop.

In [2]:
demographics = pd.read_sas("DEMO_J.xpt")
individual_foods_1day = pd.read_sas("DR1IFF_J.xpt")
individual_foods_total = pd.read_sas("DR1TOT_J.xpt")
blood_pressure = pd.read_sas("BPX_J.xpt")

The data I am pulling from the CDC website has a long list of coded variables. The definitions of these variables are not available in the individual tables themselves, and are only available in a separate list, but are not available as a downloadable table. They are available  merely to view at this link: 
https://wwwn.cdc.gov/nchs/nhanes/search/variablelist.aspx?Component=Dietary&Cycle=2017-2018
Inspecting the source code, the table found in the link is sourced from an HTML table object. This means I can use Pandas to read and download this HTML:

In [54]:
cdc_website_info = pd.read_html("https://wwwn.cdc.gov/nchs/nhanes/search/variablelist.aspx?Component=Dietary&Cycle=2017-2018")
cdc_website_info

[    Variable Name                               Variable Description  \
 0          DBD100  How often {do you/does SP} add this salt to {y...   
 1         DBQ095Z  What type of salt {do you/does SP} usually add...   
 2         DR1_300  Was the amount of food that {you/NAME} ate yes...   
 3        DR1_320Z  Total plain water drank yesterday - including ...   
 4        DR1_330Z  Total tap water drank yesterday - including fi...   
 ..            ...                                                ...   
 739        DSQIVD                          Vitamin D (D2 + D3) (mcg)   
 740        DSQIVK                                    Vitamin K (mcg)   
 741      DSQIZINC                                          Zinc (mg)   
 742       RXQ215A  Did you take {PRODUCT NAME} as an antacid, as ...   
 743          SEQN                        Respondent sequence number.   
 
     Data File Name                              Data File Description  \
 0         DR1TOT_J  Dietary Interview - Total N

Which returns this list of Data Tables. Luckily, the information I want is the first element in the list, so I can just retrieve that specific data table:

In [55]:
variable_info = cdc_website_info[0]
variable_info

,Variable Name,Variable Description,Data File Name,Data File Description,Begin Year,EndYear,Component,Use Constraints
0,DBD100,How often {do you/does SP} add this salt to {y...,DR1TOT_J,"Dietary Interview - Total Nutrient Intakes, Fi...",2017,2018,Dietary,NaN
1,DBQ095Z,What type of salt {do you/does SP} usually add...,DR1TOT_J,"Dietary Interview - Total Nutrient Intakes, Fi...",2017,2018,Dietary,NaN
2,DR1_300,Was the amount of food that {you/NAME} ate yes...,DR1TOT_J,"Dietary Interview - Total Nutrient Intakes, Fi...",2017,2018,Dietary,NaN
3,DR1_320Z,Total plain water drank yesterday - including ...,DR1TOT_J,"Dietary Interview - Total Nutrient Intakes, Fi...",2017,2018,Dietary,NaN
4,DR1_330Z,Total tap water drank yesterday - including fi...,DR1TOT_J,"Dietary Interview - Total Nutrient Intakes, Fi...",2017,2018,Dietary,NaN
...,...,...,...,...,...,...,...,...
739,DSQIVD,Vitamin D (D2 + D3) (mcg),DSQIDS_J,Dietary Supplement Use 30-Day - Individual Die...,2017,2018,Dietary,NaN
740,DSQIVK,Vitamin K (mcg),DSQIDS_J,Dietary Supplement Use 30-Day - Individual Die...,2017,2018,Dietary,NaN
741,DSQIZINC,Zinc (mg),DSQIDS_J,Dietary Supplement Use 30-Day - Individual Die...,2017,2018,Dietary,NaN
742,RXQ215A,"Did you take {PRODUCT NAME} as an antacid, as ...",DSQIDS_J,Dietary Supplement Use 30-Day - Individual Die...,2017,2018,Dietary,NaN


This table contains all of the variables found in the entire body of data. I cannot trivially verify whether variables are reused or not. To verify this information, I will first limit the rows to the information only found in the data files I am using. 

After reducing the table to the rows I want, I'll reduce the columns to just the variable names and their definitions, because that's all I need.

In [79]:
reduced_variable_info = variable_info[
variable_info.iloc[:,2].isin(['DEMO_J','DR1IFF_J','DR1TOT_J','BPX_J']) == True
    ].iloc[:,0:2]
reduced_variable_info

,Variable Name,Variable Description
0,DBD100,How often {do you/does SP} add this salt to {y...
1,DBQ095Z,What type of salt {do you/does SP} usually add...
2,DR1_300,Was the amount of food that {you/NAME} ate yes...
3,DR1_320Z,Total plain water drank yesterday - including ...
4,DR1_330Z,Total tap water drank yesterday - including fi...
...,...,...
332,DRABF,Indicates whether the sample person was an inf...
333,DRDINT,Indicates whether the sample person has intake...
334,SEQN,Respondent sequence number.
335,WTDR2D,Dietary two-day sample weight


Checking for if variables are repeated in this table:

In [75]:
variable_info['Variable Name'].value_counts()

Variable Name
SEQN        10
WTDRD1       8
DRDINT       8
WTDR2D       8
DR2EXMER     4
            ..
DSQIVC       1
DSQIVD       1
DSQIVK       1
DSQIZINC     1
DR1TALCO     1
Name: count, Length: 673, dtype: int64

Because some variables are repeated, I'll reduce this table to its unique values:

In [89]:
reduced_variable_info = reduced_variable_info.drop_duplicates(subset = ['Variable Name'])
reduced_variable_info

,Variable Name,Variable Description
0,DBD100,How often {do you/does SP} add this salt to {y...
1,DBQ095Z,What type of salt {do you/does SP} usually add...
2,DR1_300,Was the amount of food that {you/NAME} ate yes...
3,DR1_320Z,Total plain water drank yesterday - including ...
4,DR1_330Z,Total tap water drank yesterday - including fi...
...,...,...
326,DR1IVB6,Vitamin B6 (mg)
327,DR1IVC,Vitamin C (mg)
328,DR1IVD,Vitamin D (D2 + D3) (mcg)
329,DR1IVK,Vitamin K (mcg)


I now have a table containing the definition for every variable I might need. 

In [11]:
print("individual foods info:")
individual_foods_1day.info()
print("\nindividual foods total:")
individual_foods_total.info()
print("\nblood pressure:")
blood_pressure.info()
print("\ndemographics:")
demographics.info()

individual foods info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112683 entries, 0 to 112682
Data columns (total 84 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   SEQN      112683 non-null  float64
 1   WTDRD1    112683 non-null  float64
 2   WTDR2D    112683 non-null  float64
 3   DR1ILINE  112683 non-null  float64
 4   DR1DRSTZ  112683 non-null  float64
 5   DR1EXMER  112683 non-null  float64
 6   DRABF     112683 non-null  float64
 7   DRDINT    112683 non-null  float64
 8   DR1DBIH   108882 non-null  float64
 9   DR1DAY    112683 non-null  float64
 10  DR1LANG   112667 non-null  float64
 11  DR1CCMNM  112683 non-null  float64
 12  DR1CCMTX  112683 non-null  float64
 13  DR1_020   112683 non-null  float64
 14  DR1_030Z  112683 non-null  float64
 15  DR1FS     105224 non-null  float64
 16  DR1_040Z  111710 non-null  float64
 17  DR1IFDCD  112683 non-null  float64
 18  DR1IGRMS  111710 non-null  float64
 19  DR1IKCAL  111710 non-

How to read these tables:

SEQN stands for "Respondant Sequence Number" and is the ID for the specific person surveyed. The other variables reference a question asked by the survey. Now, I need to select for data that I actually need, and take care of NaN values.

One of the variables, DRD370QQ, in individual_foods_total, refers to "Number of times swordfish was eaten in the past 30 days." Looking at this variable specifically:

In [105]:
individual_foods_total.loc[:,'DRD370QQ']


0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
        ..
8699   NaN
8700   NaN
8701   NaN
8702   NaN
8703   NaN
Name: DRD370QQ, Length: 8704, dtype: float64

In [107]:
individual_foods_total.loc[:,'DRD370QQ'].value_counts()

DRD370QQ
1.0     28
2.0      3
3.0      2
20.0     1
4.0      1
Name: count, dtype: int64

The problem is immediately noticeable: The printout of that column shows an overwhelmingly large list of NaN values. Counting the value frequencies shows that it is not just a table full of NaN values, there are actual results- this leads me to believe that NaN, in this instance, is where the survey taker had not eaten swordfish in the prior 30 days, which had not simply been recorded as a "zero" value. I have two options:
1: Because the foods-eaten tables just refer to counts of specific foods eaten, I can replace all the NaN values with zeroes.
2: Because my goal is to attain information on the correlation of dietary-habits with blood pressure, I reduce the tables to the information I specifically need.
The second option would hypothetically reduce the data I am working with to only the most necessary information. However, given the question I am asking, I am obligated to operate from a blind perspective- because the goal is to find correlations between dietary habits with blood pressure, and I do not know if swordfish effects blood pressure, then deciding whether or not swordfish is statistically relevant before constructing the model is assuming the answer.

However, given that only 35 people responded with any answer to the swordfish question, I now have to resolve this secondary problem:
Is there a threshhold of responses under which the data becomes irrelevant? If only 35 people out of 7640 respond, do their responses count?

Suppose swordfish has no effect on blood pressure: 
If these people do have some other dietary habit that effects blood pressure, you would expect this habit to be consistent across whatever biological mechanism causes it, and so ideally the result would make a connection between this habit and blood pressure. For example, it is known that overconsuming sodium causes high blood pressure. Unless only the swordfish-eaters are over-consuming sodium, there should be some over-representation of sodium-overconsumers with high blood pressure that drowns-out any correlation between swordfish and high blood pressure,

Suppose swordfish massively spikes blood pressure: 
If then some other habit also spikes blood pressure, like overconsuming sodium, I would expect the same thing to happen. Suppose swordfish eaters are perfectly healthy except for their consumption of swordfish. If some other correlation is made between other habits, that the swordfish eaters do not participate in but still have high blood pressure, ideally a correlation would be made between consumption of swordfish and high blood pressure. 

The above statements are based on assumptions of low statistical variance. Over the 7640 participants, based on the law of large numbers, I expect an (approximately) normal distribution of every data point that has been answered by every participant. This is certain among foods or nutrients that are consumed by everybody, all the time, like sodium or sugars. If these foods have some effect on blood pressure, their relevance (ideally) would be noted, but would also drown out rarer food items like swordfish. As such, given an equally distributed focus of the survey-results by the model, ideally I expect the model to find patterns in the most prevalent results. 

I now need to decide, do I care if swordfish causes high blood pressure? At what level of incidence in the data do I care about a specific data point? I cannot make decisions that effect how the model responds based on my own prior knowledge. I need to assume that some accurate data might get lost. 

In [114]:
individual_foods_1day.iloc[:,0].value_counts()

SEQN
102721.0    46
96092.0     44
97236.0     42
99452.0     40
93989.0     40
            ..
100044.0     2
102523.0     1
100337.0     1
100466.0     1
99185.0      1
Name: count, Length: 7640, dtype: int64

In [96]:
individual_foods_1day

,SEQN,WTDRD1,WTDR2D,DR1ILINE,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,...,DR1IM181,DR1IM201,DR1IM221,DR1IP182,DR1IP183,DR1IP184,DR1IP204,DR1IP205,DR1IP225,DR1IP226
0,93704.0,81714.005497,82442.869214,1.0,1.0,49.0,2.0,2.0,7.0,2.0,...,4.549000e+00,4.000000e-02,5.397605e-79,3.669000e+00,4.670000e-01,5.397605e-79,2.200000e-02,5.397605e-79,1.000000e-03,6.000000e-03
1,93704.0,81714.005497,82442.869214,2.0,1.0,49.0,2.0,2.0,7.0,2.0,...,4.590000e-01,2.000000e-03,5.397605e-79,7.200000e-02,7.000000e-03,5.397605e-79,2.000000e-03,5.397605e-79,5.397605e-79,5.397605e-79
2,93704.0,81714.005497,82442.869214,3.0,1.0,49.0,2.0,2.0,7.0,2.0,...,4.656000e+00,1.060000e-01,9.000000e-03,5.485000e+00,2.260000e-01,5.397605e-79,4.000000e-02,5.397605e-79,3.000000e-03,2.000000e-03
3,93704.0,81714.005497,82442.869214,4.0,1.0,49.0,2.0,2.0,7.0,2.0,...,1.000000e-03,5.397605e-79,5.397605e-79,2.000000e-03,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
4,93704.0,81714.005497,82442.869214,5.0,1.0,49.0,2.0,2.0,7.0,2.0,...,2.043000e+00,4.100000e-02,5.397605e-79,1.418000e+00,2.630000e-01,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112678,102956.0,69447.414236,92756.884416,4.0,1.0,81.0,2.0,2.0,10.0,5.0,...,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
112679,102956.0,69447.414236,92756.884416,5.0,1.0,81.0,2.0,2.0,10.0,5.0,...,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79
112680,102956.0,69447.414236,92756.884416,6.0,1.0,81.0,2.0,2.0,10.0,5.0,...,4.317000e+00,5.700000e-02,5.397605e-79,3.086000e+00,3.190000e-01,5.397605e-79,1.700000e-01,1.100000e-02,2.200000e-02,1.400000e-02
112681,102956.0,69447.414236,92756.884416,7.0,1.0,81.0,2.0,2.0,10.0,5.0,...,4.428000e+00,2.200000e-02,5.397605e-79,7.830000e-01,1.250000e-01,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79


In [98]:
individual_foods_total

,SEQN,WTDRD1,WTDR2D,DR1DRSTZ,DR1EXMER,DRABF,DRDINT,DR1DBIH,DR1DAY,DR1LANG,...,DRD370QQ,DRD370R,DRD370RQ,DRD370S,DRD370SQ,DRD370T,DRD370TQ,DRD370U,DRD370UQ,DRD370V
0,93703.0,5.397605e-79,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,8.171401e+04,8.244287e+04,1.0,49.0,2.0,2.0,7.0,2.0,1.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0
2,93705.0,7.185561e+03,5.640391e+03,1.0,73.0,2.0,2.0,5.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,93706.0,6.463883e+03,5.397605e-79,1.0,86.0,2.0,1.0,NaN,6.0,1.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0
4,93707.0,1.533378e+04,2.270707e+04,1.0,81.0,2.0,2.0,14.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8699,102952.0,1.381296e+04,2.868593e+04,1.0,73.0,2.0,2.0,22.0,7.0,4.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0
8700,102953.0,5.063236e+04,5.397605e-79,1.0,73.0,2.0,1.0,2.0,7.0,2.0,...,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN,2.0
8701,102954.0,1.108127e+04,8.924895e+03,1.0,76.0,2.0,2.0,2.0,6.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8702,102955.0,2.752985e+04,3.629955e+04,1.0,73.0,2.0,2.0,15.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [99]:
blood_pressure

,SEQN,PEASCCT1,BPXCHR,BPAARM,BPACSZ,BPXPLS,BPXPULS,BPXPTY,BPXML1,BPXSY1,...,BPAEN1,BPXSY2,BPXDI2,BPAEN2,BPXSY3,BPXDI3,BPAEN3,BPXSY4,BPXDI4,BPAEN4
0,93703.0,NaN,120.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,NaN,114.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,NaN,NaN,1.0,4.0,52.0,1.0,1.0,220.0,NaN,...,NaN,NaN,NaN,NaN,202.0,62.0,2.0,198.0,74.0,2.0
3,93706.0,NaN,NaN,1.0,3.0,82.0,1.0,1.0,140.0,112.0,...,2.0,114.0,70.0,2.0,108.0,76.0,2.0,NaN,NaN,NaN
4,93707.0,NaN,NaN,1.0,2.0,100.0,1.0,1.0,140.0,128.0,...,2.0,128.0,46.0,2.0,128.0,58.0,2.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8699,102952.0,NaN,NaN,1.0,3.0,68.0,1.0,1.0,150.0,136.0,...,2.0,142.0,78.0,2.0,140.0,68.0,2.0,NaN,NaN,NaN
8700,102953.0,NaN,NaN,1.0,5.0,78.0,1.0,1.0,150.0,124.0,...,2.0,122.0,76.0,2.0,116.0,74.0,2.0,NaN,NaN,NaN
8701,102954.0,NaN,NaN,1.0,3.0,78.0,1.0,1.0,150.0,116.0,...,2.0,118.0,72.0,2.0,114.0,74.0,2.0,NaN,NaN,NaN
8702,102955.0,NaN,NaN,1.0,5.0,74.0,1.0,1.0,140.0,114.0,...,2.0,114.0,60.0,2.0,114.0,64.0,2.0,NaN,NaN,NaN


In [100]:
demographics

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,93703.0,10.0,2.0,2.0,2.0,NaN,5.0,6.0,2.0,27.0,...,3.0,1.0,3.0,9246.491865,8539.731348,2.0,145.0,15.0,15.0,5.00
1,93704.0,10.0,2.0,1.0,2.0,NaN,3.0,3.0,1.0,33.0,...,3.0,1.0,2.0,37338.768343,42566.614750,1.0,143.0,15.0,15.0,5.00
2,93705.0,10.0,2.0,2.0,66.0,NaN,4.0,4.0,2.0,NaN,...,1.0,2.0,NaN,8614.571172,8338.419786,2.0,145.0,3.0,3.0,0.82
3,93706.0,10.0,2.0,1.0,18.0,NaN,5.0,6.0,2.0,222.0,...,3.0,1.0,2.0,8548.632619,8723.439814,2.0,134.0,NaN,NaN,NaN
4,93707.0,10.0,2.0,1.0,13.0,NaN,5.0,7.0,2.0,158.0,...,2.0,1.0,3.0,6769.344567,7064.609730,1.0,138.0,10.0,10.0,1.88
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9249,102952.0,10.0,2.0,2.0,70.0,NaN,5.0,6.0,2.0,NaN,...,2.0,1.0,1.0,16896.276203,18338.711104,2.0,138.0,4.0,4.0,0.95
9250,102953.0,10.0,2.0,1.0,42.0,NaN,1.0,1.0,2.0,NaN,...,2.0,2.0,NaN,61630.380013,63661.951573,2.0,137.0,12.0,12.0,NaN
9251,102954.0,10.0,2.0,2.0,41.0,NaN,4.0,4.0,1.0,NaN,...,2.0,2.0,NaN,17160.895269,17694.783346,1.0,144.0,10.0,10.0,1.18
9252,102955.0,10.0,2.0,2.0,14.0,NaN,4.0,4.0,2.0,175.0,...,2.0,1.0,2.0,14238.445922,14871.839636,1.0,136.0,9.0,9.0,2.24
